In [116]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, 
    classification_report, 
    average_precision_score, 
    roc_auc_score
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')


In [117]:
# Load dataset
df = pd.read_csv('../../data/paysim/paysim.csv')
print(f"Raw dataset shape: {df.shape}")
df.head()

Raw dataset shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [118]:
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [119]:
df = df.rename(columns={
    'oldbalanceOrg': 'oldBalanceOrig',
    'newbalanceOrig': 'newBalanceOrig',
    'oldbalanceDest': 'oldBalanceDest',
    'newbalanceDest': 'newBalanceDest'
})

In [120]:
pd.crosstab(df['type'], df['isFraud'])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [121]:
df['nameOrig'].nunique()


6353307

In [122]:
df['nameDest'].nunique()

2722362

In [123]:
pd.crosstab(df['isFraud'], df['isFlaggedFraud'])

isFlaggedFraud,0,1
isFraud,,
0,6354407,0
1,8197,16


In [124]:
# Fraud onlu occurs in 'TRANSFER' and 'CASH_OUT'
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# Drop ID cols (too unique) and 'isFlaggedFraud' (ineffective)
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])

# Encode type to 0/1
df['type'] = df['type'].map({'TRANSFER': 0, 'CASH_OUT': 1})

In [125]:
df.loc[df['isFraud'] == 1, ['amount','oldBalanceOrig', 'newBalanceOrig', 'oldBalanceDest', 'newBalanceDest']].head(100)

,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest
2,181.00,181.00,0.0,0.00,0.00
3,181.00,181.00,0.0,21182.00,0.00
251,2806.00,2806.00,0.0,0.00,0.00
252,2806.00,2806.00,0.0,26202.00,0.00
680,20128.00,20128.00,0.0,0.00,0.00
...,...,...,...,...,...
47145,350705.74,350705.74,0.0,894760.68,1794033.65
49825,2539898.07,2539898.07,0.0,0.00,261290.69
49826,2539898.07,2539898.07,0.0,968.00,1842864.22
49885,361559.69,361559.69,0.0,0.00,141302.12


In [126]:
# Check how often transactions empty origin account
print((df['amount'] == df['oldBalanceOrig']).groupby(df['isFraud']).mean())

isFraud
0    0.000000
1    0.978205
dtype: float64


In [127]:
df


,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFraud
2,1,0,181.00,181.00,0.0,0.00,0.00,1
3,1,1,181.00,181.00,0.0,21182.00,0.00,1
15,1,1,229133.94,15325.00,0.0,5083.00,51513.44,0
19,1,0,215310.30,705.00,0.0,22425.00,0.00,0
24,1,0,311685.89,10835.00,0.0,6267.00,2719172.89,0
...,...,...,...,...,...,...,...,...
6362615,743,1,339682.13,339682.13,0.0,0.00,339682.13,1
6362616,743,0,6311409.28,6311409.28,0.0,0.00,0.00,1
6362617,743,1,6311409.28,6311409.28,0.0,68488.84,6379898.11,1
6362618,743,0,850002.52,850002.52,0.0,0.00,0.00,1


In [128]:
# Convert to hour of day(0-23)
df['hourOfDay'] = df['step'] % 24

# Flag transactions that wipe out origin account
df['isAccountEmptied'] = ((df['amount'] == df['oldBalanceOrig']) & (df['amount'] > 0)).astype(int)

In [129]:
# Add balance discrepancy features
df['errorBalanceOrig'] = df['newBalanceOrig'] + df['amount'] - df['oldBalanceOrig']
df['errorBalanceDest'] = df['oldBalanceDest'] + df['amount'] - df['newBalanceDest']

In [130]:
df.isnull().sum()

step                0
type                0
amount              0
oldBalanceOrig      0
newBalanceOrig      0
oldBalanceDest      0
newBalanceDest      0
isFraud             0
hourOfDay           0
isAccountEmptied    0
errorBalanceOrig    0
errorBalanceDest    0
dtype: int64

In [131]:
# Check how often transactions empty dest account
zero_dest = (df['oldBalanceDest'] == 0) & (df['newBalanceDest'] == 0) & (df['amount'] > 0)
print(pd.crosstab(zero_dest, df['isFraud'], normalize='columns'))

isFraud         0         1
row_0                      
False    0.999382  0.504444
True     0.000618  0.495556


In [132]:

zero_orig = (df['oldBalanceOrig'] == 0) & (df['newBalanceOrig'] == 0) & (df['amount'] > 0)
print(pd.crosstab(zero_orig, df['isFraud'], normalize='columns'))

isFraud         0         1
row_0                      
False    0.526268  0.996956
True     0.473732  0.003044


Quick takeaway from the crosstabs above:
1. `zero_dest` is a massive fraud signal (~50% of frauds vs <0.1% of normal trans).
2. `zero_orig` is the exact opposite (~47% of normal trans vs <0.1% of frauds)
3. Imputing both with `-1` helps the model tell the difference between untracked `(0, 0)` balances and accounts that were genuinely drained to `0`.

In [133]:
# Convert anomalous zero balances to -1 for dest
zero_dest_mask = (df['oldBalanceDest'] == 0) & (df['newBalanceDest'] == 0) & (df['amount'] != 0)
df.loc[zero_dest_mask, ['oldBalanceDest', 'newBalanceDest']] = -1

# Convert zero balances to -1 for orig for seperation
zero_orig_mask = (df['oldBalanceOrig'] == 0) & (df['newBalanceOrig'] == 0) & (df['amount'] != 0)
df.loc[zero_orig_mask, ['oldBalanceOrig', 'newBalanceOrig']] = -1

In [134]:
df


,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFraud,hourOfDay,isAccountEmptied,errorBalanceOrig,errorBalanceDest
2,1,0,181.00,181.00,0.0,-1.00,-1.00,1,1,1,0.00,1.810000e+02
3,1,1,181.00,181.00,0.0,21182.00,0.00,1,1,1,0.00,2.136300e+04
15,1,1,229133.94,15325.00,0.0,5083.00,51513.44,0,1,0,213808.94,1.827035e+05
19,1,0,215310.30,705.00,0.0,22425.00,0.00,0,1,0,214605.30,2.377353e+05
24,1,0,311685.89,10835.00,0.0,6267.00,2719172.89,0,1,0,300850.89,-2.401220e+06
...,...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,1,339682.13,339682.13,0.0,0.00,339682.13,1,23,1,0.00,0.000000e+00
6362616,743,0,6311409.28,6311409.28,0.0,-1.00,-1.00,1,23,1,0.00,6.311409e+06
6362617,743,1,6311409.28,6311409.28,0.0,68488.84,6379898.11,1,23,1,0.00,1.000000e-02
6362618,743,0,850002.52,850002.52,0.0,-1.00,-1.00,1,23,1,0.00,8.500025e+05


In [135]:

X = df.drop(columns=['isFraud'])
y = df['isFraud']

# Stratified split to keep fraud ratio consistent
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

In [136]:
num_cols = [
    'amount', 'oldBalanceOrig', 'newBalanceOrig', 
    'oldBalanceDest', 'newBalanceDest', 
    'errorBalanceOrig', 'errorBalanceDest'
]

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [137]:
log_reg = LogisticRegression(
    class_weight='balanced', 
    max_iter=1000, 
    random_state=42,
    n_jobs=-1
)

log_reg.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"n_jobs n_jobs: int, default=NoneDoes not have any effect... deprecated:: 1.8 `n_jobs` is deprecated in version 1.8 and will be removed in 1.10.",-1
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_

In [138]:
y_pred = log_reg.predict(X_test)

y_pred_proba = log_reg.predict_proba(X_test)[:, 1]
y_pred_proba

array([0.01138747, 0.00692764, 0.0047549 , ..., 0.00402446, 0.00704466,
       0.00114246], shape=(554082,))

In [ ]:


print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

roc_auc = roc_auc_score(y_test, y_pred_proba)
auprc = average_precision_score(y_test, y_pred_proba)

print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"AUPRC Score (Primary Metric): {auprc:.4f}")

Confusion Matrix:
[[552431      8]
 [     9   1634]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000    552439
           1     0.9951    0.9945    0.9948      1643

    accuracy                         1.0000    554082
   macro avg     0.9976    0.9973    0.9974    554082
weighted avg     1.0000    1.0000    1.0000    554082

ROC-AUC Score: 0.9975
AUPRC Score (Primary Metric): 0.9940
